In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("24-performance-optimization")
dfs = register_views(spark)
emp = dfs["employees"]
sal = dfs["salary_history"]
dept = dfs["departments"]

## Section 1 — Predicate pushdown (push filters early)

In [ ]:
print("=" * 70)
print("SECTION 1: Predicate Pushdown")
print("=" * 70)

# BAD: filter AFTER join (both sides fully joined first)
bad = emp.join(sal, "emp_id").filter(F.col("dept_id") == 1)
print("\n[BAD] Filter after join:")
bad.explain()

# GOOD: filter BEFORE join (reduces rows going into shuffle)
good = emp.filter(F.col("dept_id") == 1).join(sal, "emp_id")
print("\n[GOOD] Filter before join:")
good.explain()
# Note: Catalyst optimizer often rewrites the BAD form automatically; explain() confirms
print(f"Rows (GOOD): {good.count()}")

## Section 2 — Column pruning (select only needed columns early)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 2: Column Pruning")
print("=" * 70)

# BAD: select all columns then join
bad2 = emp.join(sal, "emp_id").select("emp_id", "first_name", "salary_after")
print("\n[BAD] Select after join (all columns shuffled):")
bad2.explain()

# GOOD: select before join reduces data volume in shuffle
good2 = emp.select("emp_id", "first_name", "dept_id") \
           .join(sal.select("emp_id", "salary_after", "effective_date"), "emp_id")
print("\n[GOOD] Select before join (fewer columns shuffled):")
good2.explain()
good2.show(5)

## Section 3 — Avoid UDFs when native functions exist

In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: Avoid UDFs — use native functions")
print("=" * 70)

from pyspark.sql.functions import udf

# BAD: Python UDF breaks optimizer chain, serializes per row
@udf(returnType=StringType())
def upper_udf(s):
    return s.upper() if s else None

print("\n[BAD] Python UDF (breaks optimizer, per-row Python serialization):")
emp.withColumn("upper_name", upper_udf(F.col("first_name"))).explain()

# GOOD: native function stays in JVM, optimizer can pushdown
print("\n[GOOD] Native F.upper() (stays in JVM, optimizer-friendly):")
emp.withColumn("upper_name", F.upper("first_name")).explain()
emp.withColumn("upper_name", F.upper("first_name")).select("first_name", "upper_name").show(5)

## Section 4 — Consolidate window functions with same partition+order spec

In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Reuse Window Spec (same partition+order)")
print("=" * 70)

# Define the window spec once and reuse
w = Window.partitionBy("emp_id").orderBy("effective_date")

# BAD: two separate window calls (two sort passes)
bad_w = sal.withColumn("lag_sal", F.lag("salary_after", 1).over(w)) \
           .withColumn("rn", F.row_number().over(w))  # same window spec — inefficient if separate
print("\n[BAD] Two separate window computations:")
bad_w.explain()

# GOOD: define the window spec once and reuse
good_w = sal.withColumn("lag_sal", F.lag("salary_after", 1).over(w)) \
            .withColumn("lead_sal", F.lead("salary_after", 1).over(w)) \
            .withColumn("rn", F.row_number().over(w))
# Spark usually merges same-spec windows in one pass; verify with explain()
print("\n[GOOD] Three window cols, one window spec (Spark merges in one pass):")
good_w.explain()
good_w.show(5)

## Section 5 — Broadcast join for small tables

In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Broadcast Join (small table)")
print("=" * 70)

from pyspark.sql.functions import broadcast

# GOOD: broadcast the small departments table (8 rows)
print("\n[GOOD] Broadcast small dept table — should show BroadcastHashJoin:")
emp.join(broadcast(dept), "dept_id", "left").explain()  # should show BroadcastHashJoin
emp.join(broadcast(dept), "dept_id", "left").select("emp_id", "first_name", "dept_name").show(5)

## Section 6 — Avoid collect() on large DataFrames

In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: Avoid collect() — use take(N) or show()")
print("=" * 70)

# BAD: collect() brings ALL data to driver
# emp.collect()  # risky if data is large

# GOOD: use take(N) or limit(N).collect() for sampling
print("First 3 rows:", emp.limit(3).collect())
# Or use show() which doesn't return to driver memory:
print("\nshow(3) — data never fully materialised in driver memory:")
emp.show(3)

## Section 7 — Persist strategically (avoid recomputation)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 7: Cache to Avoid Recomputation")
print("=" * 70)

# BAD: use same DataFrame in two actions without caching → recomputed twice
active = emp.filter(F.col("status") == "Active")
r1 = active.count()          # scan 1
r2 = active.agg(F.avg("salary")).collect()  # scan 2 (recomputes filter)
print(f"[BAD] active count={r1}, avg salary (recomputed): {r2}")

# GOOD: cache once, use twice
active_cached = emp.filter(F.col("status") == "Active").cache()
active_cached.count()  # materialize
r1_c = active_cached.count()                       # from cache
r2_c = active_cached.agg(F.avg("salary")).collect()  # from cache
active_cached.unpersist()
print(f"[GOOD] active count={r1_c}, avg salary (from cache): {r2_c}")

## Section 8 — Use explain() to catch unexpected plans

In [ ]:
print("\n" + "=" * 70)
print("SECTION 8: Detect Accidental Cross Join with explain()")
print("=" * 70)

# Show a "bad" cartesian product (accidental cross join):
try:
    spark.conf.set("spark.sql.crossJoin.enabled", "true")
    cross = emp.crossJoin(dept)
    print(f"Cross join rows: {cross.count()}")  # 41 * 8 = 328
    print("\nCross join plan — look for BroadcastNestedLoopJoin or CartesianProduct:")
    cross.explain()  # look for BroadcastNestedLoopJoin or CartesianProduct
finally:
    spark.conf.set("spark.sql.crossJoin.enabled", "false")

## Section 9 — coalesce after filter (right-size output partitions)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 9: coalesce() After Filter (right-size output files)")
print("=" * 70)

# After filtering to a small subset, use coalesce to reduce output files
eng_only = emp.filter(F.col("dept_id") == 1).coalesce(1)
print(f"Engineering employees: {eng_only.count()}, partitions: {eng_only.rdd.getNumPartitions()}")
out_path = output_path("parquet/eng_employees")
eng_only.write.mode("overwrite").parquet(out_path)
import os
print("Output files:", os.listdir(out_path))  # should be 1 .parquet file

stop_and_wait(spark)